In [1]:
import pandas as pd, numpy as np

BASE = "/kaggle/input/datasets/azathoth42/myanimelist/"
users = pd.read_csv(BASE + "UserList.csv", keep_default_na=False, na_values=[""])

birth = pd.to_datetime(users["birth_date"], errors="coerce")
age = (pd.Timestamp("2018-01-01") - birth).dt.days / 365.25
age = age.dropna()
print("users with a parsed birth date:", len(age))
print("born in 1930:", int((birth.dt.year == 1930).sum()))

a = age[(age >= 0) & (age <= 120)]
bands = pd.cut(a, bins=[0, 8, 10, 13, 18, 25, 35, 45, 55, 65, 75, 85, 120], right=False)
t = bands.value_counts().sort_index()
print(pd.DataFrame({"users": t, "share %": (t / len(a) * 100).round(3)}))

# without the default year 1930
a2 = age[birth.dt.year != 1930]
a2 = a2[(a2 >= 0) & (a2 <= 120)]
t2 = pd.cut(a2, bins=[0, 8, 10, 13, 18, 25, 35, 45, 55, 65, 75, 85, 120], right=False).value_counts().sort_index()
print("\nwithout birth year 1930:")
print(pd.DataFrame({"users": t2, "share %": (t2 / len(a2) * 100).round(3)}))

users with a parsed birth date: 168741
born in 1930: 842
            users  share %
birth_date                
[0, 8)        651    0.386
[8, 10)       119    0.071
[10, 13)      166    0.098
[13, 18)     9208    5.461
[18, 25)    85271   50.569
[25, 35)    67961   40.304
[35, 45)     3618    2.146
[45, 55)      426    0.253
[55, 65)      113    0.067
[65, 75)       82    0.049
[75, 85)      111    0.066
[85, 120)     896    0.531

without birth year 1930:
            users  share %
birth_date                
[0, 8)        651    0.388
[8, 10)       119    0.071
[10, 13)      166    0.099
[13, 18)     9208    5.488
[18, 25)    85271   50.823
[25, 35)    67961   40.506
[35, 45)     3618    2.156
[45, 55)      426    0.254
[55, 65)      113    0.067
[65, 75)       82    0.049
[75, 85)      111    0.066
[85, 120)      54    0.032


In [2]:
join = pd.to_datetime(users["join_date"], errors="coerce")
account_days = (pd.Timestamp("2018-05-25") - join).dt.days
users["account_days"] = account_days

cols = ["username", "user_days_spent_watching", "stats_episodes", "user_completed",
        "stats_rewatched", "join_date", "account_days"]

print("top 10 by user_days_spent_watching")
print(users.nlargest(10, "user_days_spent_watching")[cols].to_string(index=False))

print("\ntop 10 by stats_episodes")
print(users.nlargest(10, "stats_episodes")[cols].to_string(index=False))

for th in [1000, 2000, 3650, 4900, 10000]:
    print(f"days_spent_watching > {th:>6,}:", int((users["user_days_spent_watching"] > th).sum()))
for th in [20_000, 50_000, 100_000, 500_000]:
    print(f"stats_episodes > {th:>8,}:", int((users["stats_episodes"] > th).sum()))

# impossible: more days watched than days since joining
imp = users["user_days_spent_watching"] > users["account_days"]
print("\nmore days watched than days since joining:", int(imp.sum()))

top 10 by user_days_spent_watching
      username  user_days_spent_watching  stats_episodes  user_completed  stats_rewatched  join_date  account_days
       warrose                2601351.02     243794881.0            2034              0.0 2011-02-01        2670.0
     Confucius                 165535.90       3287576.0             719              0.0 2010-05-15        2932.0
dheiin_society                  41753.90       1063484.0              16             36.0 2009-08-18        3202.0
    YoruNoMajo                  31047.35       1770747.0             101              0.0 2012-02-11        2295.0
         Fovez                  30197.12       3547267.0             802            179.0 2015-05-11        1110.0
       donakos                  28091.42       1315030.0             172              2.0 2007-09-21        3899.0
 projectfear22                  25110.63       1334307.0             899              1.0 2014-02-19        1556.0
     mmmapples                  23640.42     

In [3]:
valid = {1, 2, 3, 4, 6}
parts, total = [], 0
usecols = ["username", "anime_id", "my_status", "my_score", "my_watched_episodes"]
for ch in pd.read_csv(BASE + "UserAnimeList.csv", usecols=usecols,
                      keep_default_na=False, na_values=[""], chunksize=5_000_000):
    total += len(ch)
    parts.append(ch[~ch["my_status"].isin(valid)])
bad = pd.concat(parts)
print("total rows:", f"{total:,}", "| unknown status:", len(bad))
print(bad["my_status"].value_counts())
print("users:", bad["username"].nunique())
print(bad["username"].value_counts().head(10))
print("rows with score > 0:", int((bad["my_score"] > 0).sum()),
      "| with watched episodes > 0:", int((bad["my_watched_episodes"] > 0).sum()))
print(bad.head(10).to_string(index=False))

total rows: 80,076,112 | unknown status: 537
my_status
0     527
5       6
33      3
55      1
Name: count, dtype: int64
users: 24
username
ErgoSis           338
Deazent           147
ArchAngelx240      10
hatnclogs           6
Starfola            6
rictys              4
Moelump-test        3
__Simon__           3
programmargorp      2
Snagret             2
Name: count, dtype: int64
rows with score > 0: 79 | with watched episodes > 0: 93
      username  anime_id  my_watched_episodes  my_score  my_status
     hatnclogs        21                   55        10          0
     hatnclogs       223                   24        10          0
     hatnclogs       249                    2        10          0
     hatnclogs      1604                   18        10          0
     hatnclogs     11061                   15        10          0
     hatnclogs     22043                   12        10          0
programmargorp       801                    0         0          0
programmargorp      16